# 1 Module importieren

In [10]:
import pandas as pd

import m2cgen as m2c

from scipy.stats import uniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from xgboost import XGBClassifier


# 2 Daten Importieren

## 2.1 AGMP

In [3]:
agmp_dataPath = 'datasets/agmp/'
##### Data Import and Splitting #####
agmp_data = pd.read_csv(agmp_dataPath + 'gyro_mobile.csv')    # Dataset is imbalanced with only ~1.7% of all labels being 0's
agmp_data = agmp_data.drop(columns='timestamp')
agmp_xdata = agmp_data.iloc[:,:6]
agmp_ydata = agmp_data.iloc[:,6:]

agmp_xtrain, agmp_xvaltest, agmp_ytrain, agmp_yvaltest = train_test_split( 
    agmp_xdata,
    agmp_ydata,
    random_state=0,
    train_size=0.66,
    stratify=agmp_ydata                                  # Preserve label imbalance across train- and test datasets
)

agmp_xval, agmp_xtest, agmp_yval, agmp_ytest = train_test_split( 
    agmp_xvaltest,
    agmp_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=agmp_yvaltest                                  # Preserve label imbalance across train- and test datasets
)

classratio = len(agmp_data[agmp_data['Activity']==0]) / len(agmp_data[agmp_data['Activity']==1])

agmp_ev_val = [(agmp_xval,agmp_yval)]
agmp_ev_all = [(agmp_xtrain,agmp_ytrain),(agmp_xval,agmp_yval),(agmp_xtest,agmp_ytest)]

## 2.2 HARUS

In [4]:
harus_dataPath = 'datasets/harus/'

harus_xdata = pd.read_csv(harus_dataPath + "xdata.csv", sep=";")                     
harus_ydata = pd.read_csv(harus_dataPath + "ydata.csv", sep=";")

harus_xtrain, harus_xvaltest, harus_ytrain, harus_yvaltest = train_test_split( 
    harus_xdata,
    harus_ydata,
    random_state=0,
    train_size=0.66,
    stratify=harus_ydata                                  # Preserve label imbalance across train- and test datasets
)

harus_xval, harus_xtest, harus_yval, harus_ytest = train_test_split( 
    harus_xvaltest,
    harus_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=harus_yvaltest                                  # Preserve label imbalance across train- and test datasets
)

harus_ev_val = [(harus_xval,harus_yval)]
harus_ev_all = [(harus_xtrain,harus_ytrain),(harus_xval,harus_yval),(harus_xtest,harus_ytest)]


## 2.3 SeMu

In [5]:
semu_dataPath = 'datasets/semu/'                         # Set location of dataset

semu_data = pd.read_csv(semu_dataPath + 'secondary_mushroom.csv', sep=";")    # Dataset is imbalanced with only ~1.7% of all labels being 0's
semu_xdata = semu_data.iloc[:,:20]
semu_ydata = semu_data.iloc[:,20:]    # Classratio 9:11

def encodeLabels():
    encoders = {}

    # Create a list with the names of all columns, that contain categorical data
    xdata_cats = list(semu_xdata.columns)
    [xdata_cats.remove(i) for i in ['cap-diameter','stem-height','stem-width']]

    # Encode all categorical feature-labels and save encoders in a dictionary
    for cat in xdata_cats:
        le = LabelEncoder()
        semu_xdata[cat] = le.fit_transform(semu_xdata[cat])
        encoders.update({cat:le})

    le = LabelEncoder()
    semu_ydata['class'] = le.fit_transform(semu_ydata['class'])
    encoders.update({'class':le})

    return encoders

def encodeOHE():
    xdata_categorical = semu_xdata.copy()
    numeric_features = ['cap-diameter','stem-height','stem-width']

    for i in numeric_features:
        xdata_categorical = xdata_categorical.drop(i,axis=1) 

    xdata_numeric = pd.DataFrame()
    for feat in numeric_features:
        xdata_numeric[feat] = semu_xdata[feat]

    ohe = OneHotEncoder()
    ohedata = ohe.fit_transform(xdata_categorical)

    xdata_categorical_ohe = pd.DataFrame(ohedata.toarray(), columns=ohe.get_feature_names_out(), dtype=int)

    xdata_encoded = pd.concat([xdata_numeric, xdata_categorical_ohe],axis=1)

    yEncoder = LabelEncoder()
    semu_ydata['class'] = yEncoder.fit_transform(semu_ydata['class'])

    return xdata_encoded, yEncoder

# encodeLabels()
semu_xdata, yEncoder = encodeOHE() # Weil keine Rangfolge zwischen den einzelnen Kategorien herrscht, wird OHE angewendet

semu_xtrain, semu_xvaltest, semu_ytrain, semu_yvaltest = train_test_split( 
    semu_xdata,
    semu_ydata,
    random_state=0,
    train_size=0.66,
    stratify=semu_ydata                                  # Preserve label imbalance across train- and test datasets
)

semu_xval, semu_xtest, semu_yval, semu_ytest = train_test_split( 
    semu_xvaltest,
    semu_yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=semu_yvaltest                                  # Preserve label imbalance across train- and test datasets
)

semu_ev_val = [(semu_xval,semu_yval)]
semu_ev_all = [(semu_xtrain,semu_ytrain),(semu_xval,semu_yval),(semu_xtest,semu_ytest)]

# 3 Suchraum definieren

In [ ]:
searchspace = {
    'max_depth': [1,3,5,7,9],
    'n_estimators': [100,200,300,400,500,600,700,800,900,1000],
    'min_child_weight': [1,3,5,7],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'learning_rate': uniform(0.01, 0.29),
    'gamma': uniform(0.1, 0.8)
}

# 4 Hyperparameter optimieren mit Random Search

## 4.1 AGMP

In [ ]:
agmp_xgb = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    early_stopping_rounds=10,
)

agmp_random_search = RandomizedSearchCV(
    estimator=agmp_xgb, 
    param_distributions=searchspace, 
    scoring='balanced_accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
agmp_random_search.fit(
    agmp_xtrain, agmp_ytrain,
    eval_set=agmp_ev_val,
    verbose=True
)

# Print best parameters
print(f"Best parameters: {agmp_random_search.best_params_}")
print(f"Best score: {agmp_random_search.best_score_}")

## 4.2 HARUS


In [ ]:
harus_xgb = XGBClassifier(
    objective='multi:softmax',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10
)

harus_random_search = RandomizedSearchCV(
    estimator=harus_xgb, 
    param_distributions=searchspace, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
harus_random_search.fit(harus_xtrain, harus_ytrain,
                  eval_set = harus_ev_val,
                  verbose=False)

# Print best parameters
print(f"Best parameters: {harus_random_search.best_params_}")
print(f"Best score: {harus_random_search.best_score_}")

## 4.3 SeMu

In [ ]:
semu_xgb = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
)

semu_random_search = RandomizedSearchCV(
    estimator=semu_xgb, 
    param_distributions=searchspace, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
semu_random_search.fit(
    semu_xtrain, semu_ytrain,
    eval_set=semu_ev_val,
    verbose=True
)

# Print best parameters
print(f"Best parameters: {semu_random_search.best_params_}")
print(f"Best score: {semu_random_search.best_score_}")


# 5 Modelle mit optimierten Hyperparametern erzeugen

## 5.1 AGMP


In [6]:
pre_agmp = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
    scale_pos_weight=classratio,
    n_estimators=100,
    max_depth=3,
    min_child_weight=1,
    subsample=0.6352992958289331,
    colsample_bytree=0.8712237640848051,
    learning_rate=0.05873233119671418,
    gamma=0.2157529115709422,
)

pre_agmp.fit(
    agmp_xtrain, 
    agmp_ytrain,
    eval_set=agmp_ev_val,
    verbose=False
)

agmp_best_iteration = pre_agmp.best_iteration
print(f'n_estimators nach best_iteration: {agmp_best_iteration}')

agmp = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    n_estimators=agmp_best_iteration,
    max_depth=3,
    min_child_weight=1,
    subsample=0.6352992958289331,
    colsample_bytree=0.8712237640848051,
    learning_rate=0.05873233119671418,
    gamma=0.2157529115709422,
)

agmp.fit(
    agmp_xtrain, agmp_ytrain
)

n_estimators nach best_iteration: 99


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8712237640848051, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.2157529115709422,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05873233119671418,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=99, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

## 5.2 HARUS


In [7]:
pre_harus = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    early_stopping_rounds = 10,
    n_jobs = -1,
    n_estimators = 1000,
    max_depth = 3,
    min_child_weight = 1,
    subsample = 0.6575317462724224,
    colsample_bytree = 0.8582280977824015,
    learning_rate = 0.13481670745733776,
    gamma = 0.12828994860439275,
)

pre_harus.fit(
    harus_xtrain, 
    harus_ytrain,
    eval_set=harus_ev_val,
    verbose=False
)

harus_best_iteration = pre_harus.best_iteration
print(f'n_estimators nach best_iteration: {harus_best_iteration}')

harus = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    n_jobs = -1,
    n_estimators = harus_best_iteration,
    max_depth = 3,
    min_child_weight = 1,
    subsample = 0.6575317462724224,
    colsample_bytree = 0.8582280977824015,
    learning_rate = 0.13481670745733776,
    gamma = 0.12828994860439275,
)

harus.fit(
    harus_xtrain, harus_ytrain
)

n_estimators nach best_iteration: 255


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8582280977824015, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.12828994860439275,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.13481670745733776,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=255, n_jobs=-1,
              num_parallel_tree=None, objective='multi:softmax', ...)

## 5.3 SeMu

In [9]:
pre_semu = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10,
    n_estimators = 900,
    max_depth = 9,
    min_child_weight = 1, 
    learning_rate = 0.2943352730637827,
    subsample = 0.6791931199759179,
    colsample_bytree = 0.919231607902399, 
    gamma = 0.540795176148389, 
)

pre_semu.fit(
    semu_xtrain, 
    semu_ytrain,
    eval_set=semu_ev_val,
    verbose=False
)

semu_best_iteration = pre_semu.best_iteration
print(f'n_estimators nach best_iteration: {semu_best_iteration}')

semu = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    n_jobs=-1,
    n_estimators = semu_best_iteration,
    max_depth = 9,
    min_child_weight = 1, 
    learning_rate = 0.2943352730637827,
    subsample = 0.6791931199759179,
    colsample_bytree = 0.919231607902399, 
    gamma = 0.540795176148389,     
)
semu.fit(
    semu_xtrain, semu_ytrain
)

n_estimators nach best_iteration: 121


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.919231607902399, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.540795176148389,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2943352730637827,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=121, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

# 6 Modelle exportieren und portieren

## 6.1 AGMP

In [13]:
agmp.save_model("model-exports/aktuelle-exports/hyperparam_agmp.json")

agmp.set_params(base_score = 0.49990478)

with open("model-exports/plainC/hyperparam/hyperparam_agmp.c","w") as f:
    code = m2c.export_to_c(agmp)
    f.write(code)

print(f'Model exported to: "model-exports/plainC/hyperparam/hyperparam_agmp.c"')

Model exported to: "model-exports/plainC/hyperparam/hyperparam_agmp.c"


## 6.2 HARUS

In [15]:
harus.save_model("model-exports/aktuelle-exports/hyperparam_harus.json")

harus.set_params(
    base_score = 0.5,
    num_parallel_tree = 1
)

with open("model-exports/plainC/hyperparam/hyperparam_harus.c","w") as f:
    code = m2c.export_to_c(harus)
    f.write(code)

print(f'Model exported to: "model-exports/plainC/hyperparam/hyperparam_harus.c"')

Model exported to: "model-exports/plainC/hyperparam/hyperparam_harus.c"


## 6.3 SeMu

In [17]:
semu.save_model("model-exports/aktuelle-exports/hyperparam_semu.json") 

semu.set_params(base_score = 0.55469894)

with open("model-exports/plainC/hyperparam/hyperparam_semu.c","w") as f:
    code = m2c.export_to_c(semu)
    f.write(code)

print(f'Model exported to: "model-exports/plainC/hyperparam/hyperparam_semu.c"')

Model exported to: "model-exports/plainC/hyperparam/hyperparam_semu.c"
